# AI-Powered Network Traffic Analysis Using PandasAI and Power BI

Descriptive analysis of the **NSL-KDD** intrusion dataset with Pandas, natural-language questions with PandasAI, and CSV exports for Power BI Desktop.

No machine learning, prediction or live monitoring is used. NSL-KDD has no IP addresses or timestamps, so none appear here.

## 0. Setup
Run locally from the `notebooks` folder, or in Google Colab (upload the project folder, or run the Colab cell below).

In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/network-security-analysis")
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(PROJECT / "src"))
import pandas as pd
pd.set_option("display.max_columns", 50)
print("Project folder:", PROJECT)

Project folder: /home/claude/nsa


## 1. Load the dataset
The files have **no header row**, so the 41 standard NSL-KDD feature names plus `label` and `difficulty_level` are applied. Train and test are kept separate; choose one with `SOURCE`.

In [2]:
from data_loader import load_dataset, read_nslkdd, DATA_DIR, DATASET_FILES
import analysis as A

SOURCE = "train"
raw = read_nslkdd(DATA_DIR / DATASET_FILES[SOURCE])
df, report = load_dataset(SOURCE)
print("Dataset:", SOURCE)
print("Shape:", raw.shape)
raw.head()

Dataset: train
Shape: (125973, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.00,0.00,0.00,150,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,1,0.0,0.0,0.0,0.0,0.08,0.15,0.00,255,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,6,1.0,1.0,0.0,0.0,0.05,0.07,0.00,255,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,5,0.2,0.2,0.0,0.0,1.00,0.00,0.00,30,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,30,32,0.0,0.0,0.0,0.0,1.00,0.00,0.09,255,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [3]:
raw.dtypes.value_counts()

int64      24
float64    15
object      4
Name: count, dtype: int64

## 2. Data quality: missing values and duplicates

In [4]:
print(report)
quality = A.data_quality(raw)
quality[quality.missing_values > 0] if quality.missing_values.any() else print("No missing values in any column.")

{'dataset_source': 'train', 'rows_in_file': 125973, 'missing_values_in_file': 0, 'duplicate_rows_in_file': 0, 'rows_removed_missing_or_invalid': 0, 'has_difficulty_level': True, 'rows_after_cleaning': 125973, 'unknown_attack_labels': []}
No missing values in any column.


## 3. Derived columns
- `traffic_type`: Normal or Attack
- `is_attack`: 1/0
- `attack_family`: DoS, Probe, R2L, U2R, or **Unknown** for labels outside the mapping (`src/attack_families.json`)

The original `label` is kept.

In [5]:
df[['protocol_type', 'service', 'flag', 'label', 'traffic_type', 'attack_family']].head(10)

,protocol_type,service,flag,label,traffic_type,attack_family
0,tcp,ftp_data,SF,normal,Normal,Normal
1,udp,other,SF,normal,Normal,Normal
2,tcp,private,S0,neptune,Attack,DoS
3,tcp,http,SF,normal,Normal,Normal
4,tcp,http,SF,normal,Normal,Normal
5,tcp,private,REJ,neptune,Attack,DoS
6,tcp,private,S0,neptune,Attack,DoS
7,tcp,private,S0,neptune,Attack,DoS
8,tcp,remote_job,S0,neptune,Attack,DoS
9,tcp,private,S0,neptune,Attack,DoS


## 4. Overview

In [6]:
A.overview(df)

{'dataset_source': 'train',
 'total_connections': 125973,
 'normal_connections': 67343,
 'attack_connections': 58630,
 'attack_percentage': 46.54,
 'distinct_attack_labels': 22,
 'most_frequent_protocol': 'tcp'}

In [7]:
A.normal_vs_attack(df)

,traffic_type,connections,percentage_of_total
0,Normal,67343,53.46
1,Attack,58630,46.54


## 5. Attack labels and families

In [8]:
A.attack_family_summary(df)

,attack_family,connections,distinct_labels,percentage_of_attacks,percentage_of_total
0,DoS,45927,6,78.33,36.46
1,Probe,11656,4,19.88,9.25
2,R2L,995,8,1.70,0.79
3,U2R,52,4,0.09,0.04


In [9]:
A.attack_distribution(df).head(15)

,attack_label,attack_family,connections,percentage_of_attacks,percentage_of_total
0,neptune,DoS,41214,70.30,32.72
1,satan,Probe,3633,6.20,2.88
2,ipsweep,Probe,3599,6.14,2.86
3,portsweep,Probe,2931,5.00,2.33
4,smurf,DoS,2646,4.51,2.10
5,nmap,Probe,1493,2.55,1.19
6,back,DoS,956,1.63,0.76
7,teardrop,DoS,892,1.52,0.71
8,warezclient,R2L,890,1.52,0.71
9,pod,DoS,201,0.34,0.16


## 6. Protocols, services and flags
`attack_percentage_within` = attacks of that value ÷ all records of that value.

In [10]:
A.category_summary(df, 'protocol_type')

,protocol_type,total_connections,normal_connections,attack_connections,percentage_of_total,attack_percentage_within,percentage_of_all_attacks
0,tcp,102689,53600,49089,81.52,47.80,83.73
1,udp,14993,12434,2559,11.90,17.07,4.36
2,icmp,8291,1309,6982,6.58,84.21,11.91


In [11]:
A.category_summary(df, 'service').head(15)

,service,total_connections,normal_connections,attack_connections,percentage_of_total,attack_percentage_within,percentage_of_all_attacks
0,http,40338,38049,2289,32.02,5.67,3.90
1,private,21853,982,20871,17.35,95.51,35.60
2,domain_u,9043,9034,9,7.18,0.10,0.02
3,smtp,7313,7029,284,5.81,3.88,0.48
4,ftp_data,6860,4984,1876,5.45,27.35,3.20
5,eco_i,4586,497,4089,3.64,89.16,6.97
6,other,4359,2604,1755,3.46,40.26,2.99
7,ecr_i,3077,190,2887,2.44,93.83,4.92
8,telnet,2353,917,1436,1.87,61.03,2.45
9,finger,1767,545,1222,1.40,69.16,2.08


In [12]:
A.category_summary(df, 'flag')

,flag,total_connections,normal_connections,attack_connections,percentage_of_total,attack_percentage_within,percentage_of_all_attacks
0,SF,74945,63393,11552,59.49,15.41,19.70
1,S0,34851,354,34497,27.67,98.98,58.84
2,REJ,11233,2693,8540,8.92,76.03,14.57
3,RSTR,2421,146,2275,1.92,93.97,3.88
4,RSTO,1562,219,1343,1.24,85.98,2.29
5,S1,365,361,4,0.29,1.10,0.01
6,SH,271,2,269,0.22,99.26,0.46
7,S2,127,119,8,0.10,6.30,0.01
8,RSTOS0,103,0,103,0.08,100.00,0.18
9,S3,49,45,4,0.04,8.16,0.01


In [13]:
A.attack_breakdown(df, 'service').head(15)

,service,attack_family,attack_connections,percentage_of_attacks
0,private,DoS,15971,27.24
1,private,Probe,4900,8.36
2,eco_i,Probe,4089,6.97
3,ecr_i,DoS,2844,4.85
4,http,DoS,2255,3.85
5,other,Probe,1689,2.88
6,telnet,DoS,1312,2.24
7,ftp_data,DoS,1209,2.06
8,finger,DoS,1168,1.99
9,Z39_50,DoS,851,1.45


## 7. Duration and byte distributions
These features are heavily skewed (most values are 0), so the histograms use log(1 + x).

In [14]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["duration", "src_bytes", "dst_bytes"]):
    for traffic, color in [("Normal", "tab:green"), ("Attack", "tab:red")]:
        ax.hist(np.log1p(df.loc[df.traffic_type == traffic, col]), bins=40, alpha=0.6, label=traffic, color=color)
    ax.set_title(f"log(1 + {col})")
    ax.legend()
plt.tight_layout()
plt.show()

<Figure size 1500x400 with 3 Axes>

## 8. Numeric summaries and normal vs attack comparison
Differences describe this dataset only; they are associations, not causes.

In [15]:
A.numeric_feature_summary(df).query("traffic_type == 'All'").head(15)

,traffic_type,feature,count,mean,std,min,q25,median,q75,max,zero_percentage
0,All,duration,125973,287.1447,2.604515e+03,0.0,0.0,0.0,0.0,4.290800e+04,92.05
1,All,src_bytes,125973,45566.7430,5.870331e+06,0.0,0.0,44.0,276.0,1.379964e+09,39.21
2,All,dst_bytes,125973,19779.1144,4.021269e+06,0.0,0.0,0.0,516.0,1.309937e+09,53.95
3,All,land,125973,0.0002,1.410000e-02,0.0,0.0,0.0,0.0,1.000000e+00,99.98
4,All,wrong_fragment,125973,0.0227,2.535000e-01,0.0,0.0,0.0,0.0,3.000000e+00,99.13
5,All,urgent,125973,0.0001,1.440000e-02,0.0,0.0,0.0,0.0,3.000000e+00,99.99
6,All,hot,125973,0.2044,2.150000e+00,0.0,0.0,0.0,0.0,7.700000e+01,97.88
7,All,num_failed_logins,125973,0.0012,4.520000e-02,0.0,0.0,0.0,0.0,5.000000e+00,99.90
8,All,logged_in,125973,0.3957,4.890000e-01,0.0,0.0,0.0,1.0,1.000000e+00,60.43
9,All,num_compromised,125973,0.2793,2.394200e+01,0.0,0.0,0.0,0.0,7.479000e+03,98.98


In [16]:
A.compare_normal_attack(df)

traffic_type              Attack     Normal
duration       mean      423.321    168.587
               median      0.000      0.000
src_bytes      mean    82820.141  13133.279
               median      0.000    233.000
dst_bytes      mean    37524.482   4329.685
               median      0.000    379.000
count          mean      154.850     22.518
               median    138.000      4.000
srv_count      mean       27.798     27.686
               median     10.000      5.000
serror_rate    mean        0.596      0.013
               median      1.000      0.000
same_srv_rate  mean        0.307      0.969
               median      0.080      1.000
dst_host_count mean      222.025    147.432
               median    255.000    156.000

## 9. Export CSV files for Power BI
Writes both train and test into `exports/`, each row tagged with `dataset_source`.

In [17]:
from export_data import build_exports, write_exports, EXPORT_DIR

datasets = {s: load_dataset(s)[0] for s in ["train", "test"]}
for p in write_exports(build_exports(datasets)):
    print(p.name, len(pd.read_csv(p)), "rows")

cleaned_dataset.csv 148517 rows
normal_vs_attack.csv 4 rows
attack_distribution.csv 59 rows
attack_family_summary.csv 8 rows
protocol_summary.csv 6 rows
service_summary.csv 134 rows
flag_summary.csv 22 rows
protocol_attack_summary.csv 19 rows
service_attack_summary.csv 247 rows
numeric_feature_summary.csv 228 rows


## 10. PandasAI: ask questions in plain English
Requires Python 3.10/3.11, `pip install -r requirements-pandasai.txt`, and `PANDASAI_LLM_MODEL` plus an API key in `.env`.

**Data sharing:** the question, column names, table size and the first 5 rows are sent to the LLM provider. Each PandasAI answer is labelled and checked against the Pandas result where possible.

In [18]:
import pandasai_analysis as P

status = P.configuration_status()
print(status)
print(P.DATA_SHARING_NOTE)

{'installed': True, 'model': None, 'key_env': 'OPENAI_API_KEY', 'ready': False, 'problems': ['PANDASAI_LLM_MODEL is not set in .env.']}
PandasAI sends your question, the column names and types, the table size and the first 5 rows of the dataset to the configured LLM provider so it can write code. The full dataset stays on your computer; the generated SQL runs locally.


In [19]:
for q in P.EXAMPLE_QUESTIONS:
    ref = P.pandas_reference(q, df)
    print(f"[Pandas] {q} -> {ref['value'] if ref else 'see tables above'}")

[Pandas] How many DoS attacks are present? -> 45927
[Pandas] Which protocol occurs most frequently? -> tcp
[Pandas] What percentage of connections are attacks? -> 46.54
[Pandas] Show the distribution of attack types. -> see tables above
[Pandas] Which service has the highest number of attacks? -> private
[Pandas] Compare normal and attack connections. -> see tables above
[Pandas] What is the average duration for normal and attack connections? -> {'Attack': 423.3207, 'Normal': 168.5874}
[Pandas] Summarize the dataset. -> see tables above
[Pandas] Which attack family is most frequent? -> DoS


In [20]:
if status["ready"]:
    for q in ["How many DoS attacks are present?", "Which service has the highest number of attacks?"]:
        result = P.ask(q, df)
        print("Q:", q)
        if "error" in result:
            print("  ", result["error"])
        else:
            print(f"  [{result['source']}]", result["answer"])
            print("   Check:", result["pandas_check"])
else:
    print("PandasAI not configured:", status["problems"])

PandasAI not configured: ['PANDASAI_LLM_MODEL is not set in .env.']
